# DEG - Thüringen

In [1]:
import os

import patoolib
import pandas as pd
from tqdm import tqdm
import pyproj
from camelsp import get_metadata

from camels_de1h import get_input_path, get_nuts_id_from_provider_id, Station1h

## Extract data

* Stand September 2025: 
    * 7 Pegel Maingebiet
    * 8 Pegel Werragebiet
    * 33 Pegel Zuflüsse Werra-/Leinegebiet
* 40 Pegel sollen wir wohl mindestens erhalten

In [2]:
input_path = get_input_path("DEG")

# extract 7z
if not (input_path / "Q_Maingebiet").exists():
    os.makedirs(input_path / "Q_Maingebiet")
    patoolib.extract_archive(str(input_path / "TH_Pegel-Maingebiet.7z"), outdir=str(input_path / "Q_Maingebiet"), interactive=False, verbosity=-1)
    print("Maingebiet data extracted.")
else:
    print("Maingebiet data already extracted.")

if not (input_path / "Q_Werragebiet").exists():
    os.makedirs(input_path / "Q_Werragebiet")
    patoolib.extract_archive(str(input_path / "TH_Pegel-Werrafluss.7z"), outdir=str(input_path / "Q_Werragebiet"), interactive=False, verbosity=-1)
    print("Werragebiet data extracted.")
else:
    print("Werragebiet data already extracted.")

if not (input_path / "Q_Werra_Leinegebiet").exists():
    os.makedirs(input_path / "Q_Werra_Leinegebiet")
    patoolib.extract_archive(str(input_path / "ZuflüsseWerra-LeinegebietTH.7z"), outdir=str(input_path / "Q_Werra_Leinegebiet"), interactive=False, verbosity=-1)
    print("Werra-/Leinegebiet data extracted.")
else:
    print("Werra-/Leinegebiet data already extracted.")

if not (input_path / "Q_Saalegebiet").exists():
    os.makedirs(input_path / "Q_Saalegebiet")
    patoolib.extract_archive(str(input_path / "SaalegebietTH.7z"), outdir=str(input_path / "Q_Saalegebiet"), interactive=False, verbosity=-1)
    print("Saalegebiet data extracted.")
else:
    print("Saalegebiet data already extracted.")

Maingebiet data already extracted.
Werragebiet data already extracted.
Werra-/Leinegebiet data already extracted.
Saalegebiet data already extracted.


ID for station Weida-Heinoldsmühle is wrong in Saale metadata (should be 577350) --> corrected manually in the Excel file!

## Parse data

**Timezone: UTC+1 (MEZ)**

Data is in 15 minute resolution.

In [3]:
raw_meta_main = pd.read_excel(input_path / "Q_Maingebiet" / "Stammdaten_TH_Pegel-Maingebiet.xlsx")
raw_meta_main["PegelID"] = raw_meta_main["PegelID"].astype(str)


raw_meta_werra = pd.read_excel(input_path / "Q_Werragebiet" / "Stammdaten_TH_Pegel-Werrafluss.xlsx")
raw_meta_werra["PegelID"] = raw_meta_werra["PegelID"].astype(str)

raw_meta_werra_leine = pd.read_excel(input_path / "Q_Werra_Leinegebiet" / "Stammdaten_TH_Pegel-Werrazuflüsse-Leinegebiet.xlsx")
raw_meta_werra_leine["PegelID"] = raw_meta_werra_leine["PegelID"].astype(str)

raw_meta_saale = pd.read_excel(input_path / "Q_Saalegebiet" / "Stammdaten_TH_Pegel-Saalegebiet.xlsx")
raw_meta_saale["PegelID"] = raw_meta_saale["PegelID"].astype(str)

# Combine metadata
raw_meta = pd.concat([raw_meta_main, raw_meta_werra, raw_meta_werra_leine, raw_meta_saale], ignore_index=True)

ids = raw_meta["PegelID"].values

raw_meta

,PegelID,Name,Gewässer,Länge,Breite,Pegelnull [DHHN2016],AE [km²],Daten ab,Anmerkung
0,252401,Steinach,Steinach,11.1589,50.4304,485.530,37.2,1995-11-30,NaN
1,252450,Mupperg,Steinach,11.1416,50.2969,322.160,138.0,1997-11-13,Hochwasser Februar 1997 eingearbeitet; 1.7.-11...
2,252460,Hüttengrund,Engnitz,11.2043,50.4043,445.160,42.1,1998-12-02,NaN
3,252500,Almerswind-Itz,Itz,11.0084,50.3703,359.090,44.2,1996-11-01,NaN
4,252510,Almerswind,Grümpen,11.0123,50.3702,362.565,32.3,1996-11-01,NaN
...,...,...,...,...,...,...,...,...,...
135,577350,Weida-Heinoldsmühle,Auma,12.0159,50.7681,271.878,106.9,2012-11-01,NaN
136,577360,Eisenhammer (AP),Auma,12.0316,50.7649,257.052,136.2,2012-11-01,Talsperrenbeeinflusst
137,577400,Thieschitz,Erlbach,12.0420,50.9043,184.931,108.8,2012-11-01,NaN
138,577510,Gößnitz,Pleiße,12.4305,50.8911,202.182,293.0,2012-11-01,NaN


A seperate file with station locations was provided, as the precission of the lon / lat coordinates in the metadata above is quite coarse (2-4 digits).

In [20]:
df_locations = pd.read_excel(input_path / "Pegelstandorte.xlsx")

# make pegelid from e.g. 25240.1 to 252401
df_locations["PegelID"] = df_locations["pegelid"].apply(lambda x: str(x).replace(".", ""))

# transform coordinates to EPSG:4326 and EPSG:3035
# parse location
x, y = df_locations["utm_rechts"], df_locations["utm_hoch"]

# from EPSG:25832 to EPSG:3035
transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:3035", always_xy=True)
easting, northing = transformer.transform(x, y)

# from EPSG:25832 to EPSG:4326
transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:4326", always_xy=True)
lon, lat = transformer.transform(x, y)

# add to dataframe
df_locations["easting"], df_locations["northing"] = easting, northing
df_locations["lon"], df_locations["lat"] = lon, lat

# merge with raw_meta
raw_meta = raw_meta.merge(df_locations[["PegelID", "easting", "northing", "lon", "lat"]], on="PegelID", how="left")

raw_meta

,PegelID,Name,Gewässer,Länge,Breite,Pegelnull [DHHN2016],AE [km²],Daten ab,Anmerkung,easting,northing,lon,lat
0,252401,Steinach,Steinach,11.1589,50.4304,485.530,37.2,1995-11-30,NaN,4.403342e+06,3.036039e+06,11.158852,50.430459
1,252450,Mupperg,Steinach,11.1416,50.2969,322.160,138.0,1997-11-13,Hochwasser Februar 1997 eingearbeitet; 1.7.-11...,4.402663e+06,3.021175e+06,11.146061,50.296945
2,252460,Hüttengrund,Engnitz,11.2043,50.4043,445.160,42.1,1998-12-02,NaN,4.406621e+06,3.033185e+06,11.204333,50.404331
3,252500,Almerswind-Itz,Itz,11.0084,50.3703,359.090,44.2,1996-11-01,NaN,4.392743e+06,3.029186e+06,11.008396,50.370265
4,252510,Almerswind,Grümpen,11.0123,50.3702,362.565,32.3,1996-11-01,NaN,4.393023e+06,3.029183e+06,11.012332,50.370206
...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,577350,Weida-Heinoldsmühle,Auma,12.0159,50.7681,271.878,106.9,2012-11-01,NaN,4.463203e+06,3.074903e+06,12.015941,50.768096
136,577360,Eisenhammer (AP),Auma,12.0316,50.7649,257.052,136.2,2012-11-01,Talsperrenbeeinflusst,4.464316e+06,3.074578e+06,12.031593,50.764901
137,577400,Thieschitz,Erlbach,12.0420,50.9043,184.931,108.8,2012-11-01,NaN,4.464620e+06,3.090102e+06,12.041997,50.904309
138,577510,Gößnitz,Pleiße,12.4305,50.8911,202.182,293.0,2012-11-01,NaN,4.491985e+06,3.089462e+06,12.430532,50.891045


In [21]:
camelsp_meta_all = get_metadata()
camelsp_meta_all[camelsp_meta_all["provider_id"].isin(ids)].iloc[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]]

,camels_id,provider_id,camels_path,nuts_lvl2,federal_state,gauge_name,waterbody_name,gauge_elevation,area,x,y,lon,lat,q_count,w_count,q_extent_years,w_extent_years,q_w_pearson
0,DEG10000,573000,./DEG/DEG10000/DEG10000_data.csv,DEG,Thüringen,Ammern,Unstrut,210.243,182.7,4.352221e+06,3.124617e+06,10.446993,51.231727,29646.0,29646.0,81.219178,32.186301,0.969240
1,DEG10010,447000,./DEG/DEG10010/DEG10010_data.csv,DEG,Thüringen,Arenshausen,Leine,196.288,275.0,4.318941e+06,3.140875e+06,9.970428,51.378709,22707.0,22707.0,62.208219,59.876712,0.709148
2,DEG10020,574200,./DEG/DEG10020/DEG10020_data.csv,DEG,Thüringen,Arnstadt,Gera,293.577,174.7,4.386764e+06,3.077926e+06,10.933022,50.809106,35490.0,35490.0,97.230137,32.186301,0.958767
3,DEG10030,576500,./DEG/DEG10030/DEG10030_data.csv,DEG,Thüringen,Berga,Weiße Elster,218.995,1383.0,4.473276e+06,3.073272e+06,12.157989,50.750857,12845.0,12845.0,31.186301,35.189041,0.502141
4,DEG10040,570210,./DEG/DEG10040/DEG10040_data.csv,DEG,Thüringen,Blankenstein-Rosenthal,Saale,410.517,1013.0,4.442190e+06,3.033884e+06,11.704738,50.404273,21246.0,21246.0,58.205479,52.032877,0.940139
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,DEG10580,427010,./DEG/DEG10580/DEG10580_data.csv,DEG,Thüringen,Unterbreizbach-Räsa,Ulster,233.323,399.0,4.319358e+06,3.077271e+06,9.976701,50.806986,29646.0,29646.0,81.219178,80.219178,0.762258
59,DEG10590,420120,./DEG/DEG10590/DEG10590_data.csv,DEG,Thüringen,Vacha,Werra,222.678,2246.0,4.324359e+06,3.080275e+06,10.047673,50.833979,36586.0,36586.0,100.232877,85.221918,0.330791
60,DEG10600,575110,./DEG/DEG10600/DEG10600_data.csv,DEG,Thüringen,Wasserthaleben,Helbe,174.317,374.3,4.383231e+06,3.127721e+06,10.891459,51.257069,21976.0,21976.0,60.205479,60.205479,0.700221
61,DEG10610,577320,./DEG/DEG10610/DEG10610_data.csv,DEG,Thüringen,Weida,Weida,238.358,296.7,4.466471e+06,3.074267e+06,12.061997,50.761560,36221.0,36221.0,99.232877,51.200000,0.917030


Check new and old (CAMELS-DE daily) metadata.

In [23]:
for id in raw_meta["PegelID"].values:
    if id in camelsp_meta_all["provider_id"].values:
        new = raw_meta[raw_meta["PegelID"]==id]
        old = camelsp_meta_all[camelsp_meta_all["provider_id"]==id]
        
        # check columns
        if new["Name"].values[0] != old["gauge_name"].values[0]:
            print(f"{id} - {new['Name'].values[0]} != {old['gauge_name'].values[0]}")

        if new["Gewässer"].values[0] != old["waterbody_name"].values[0]:
            print(f"{id} - {new['Gewässer'].values[0]} != {old['waterbody_name'].values[0]}")

        if (new["lon"].values[0].round(2), new["lat"].values[0].round(2)) != (old["lon"].values[0].round(2), old["lat"].values[0].round(2)):
            print(f"{id} - {new['Länge'].values[0], new['Breite'].values[0]} != {old['lon'].values[0], old['lat'].values[0]}")

        if new["AE [km²]"].values[0] != old["area"].values[0]:
            print(f"{id} - {new['AE [km²]'].values[0]} != {old['area'].values[0]}")

        if new["Pegelnull [DHHN2016]"].values[0] != old["gauge_elevation"].values[0]:
            print(f"{id} - {new['Pegelnull [DHHN2016]'].values[0]} != {old['gauge_elevation'].values[0]}")

420001 - Eisfeld != Eisfeld.Bahnbrücke
420190 - 4214.0 != 4214.4
422000 - Ellinghausen != Ellingshausen
427010 - Unterbreizbach != Unterbreizbach-Räsa
427010 - 214.0 != 399.0
427010 - 233.02 != 233.323
429050 - 105.2 != 105.0
570250 - Kaulsdorf (AP) != Kaulsdorf
572110 - Katzhütte-Schwarza != Katzhütte
573000 - 183.0 != 182.7
573100 - HRB Straußfurt (AP) != Straußfurt
574620 - Georgenthal 1 != Georgenthal1
575110 - 374.0 != 374.3
575250 - 223.767 != 223.76
575400 - 201.0 != 200.6
575500 - 304.0 != 303.6
575660 - Beere != Bere
577290 - Läwitz (ZP) != Läwitz
577360 - Eisenhammer (AP) != Eisenhammer


Only real difference is area of Unterbreizbach(-Räsa), internet points to 399.0 km², which would make the old metadata correct.  

The stations mainly have the same metadata as in CAMELS-DE daily, so we just use the old metadata for all stations.

In [24]:
def parse_station_data(id: str, aggregate_hourly: bool, gebiet: str) -> pd.DataFrame:    
    # set the data path
    if gebiet == "Maingebiet":
        path = input_path / "Q_Maingebiet" / f"{id}.txt"
    elif gebiet == "Werragebiet":
        path = input_path / "Q_Werragebiet" / f"{id}.csv"
    elif gebiet == "Werra_Leinegebiet":
        path = input_path / "Q_Werra_Leinegebiet" / f"{id}_Q.txt"
    elif gebiet == "Saalegebiet":
        path = input_path / "Q_Saalegebiet" / f"{id}_Q.txt"
    else:
        raise ValueError("gebiet must be either 'Maingebiet', 'Werragebiet', 'Werra_Leinegebiet' or 'Saalegebiet'.")
    
    # read q data for the station
    data = pd.read_csv(path, sep="[;\t]", decimal=",", skiprows=3, 
                        encoding="iso-8859-1", engine='python',
                        parse_dates=["Datum Uhrzeit"], date_format="%d.%m.%Y %H:%M")
    data = data.rename(columns={"Datum Uhrzeit": "date", "Q": "discharge_vol_obs"})
    data = data[["date", "discharge_vol_obs"]] # keep only time and discharge columns

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    # add empty water level column
    data["water_level_obs"] = None

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    return data

parse_station_data(ids[93], aggregate_hourly=True, gebiet="Saalegebiet")

,date,discharge_vol_obs,water_level_obs
0,2018-10-31 23:00:00+00:00,1.34,NaN
1,2018-11-01 00:00:00+00:00,1.34,NaN
2,2018-11-01 01:00:00+00:00,1.34,NaN
3,2018-11-01 02:00:00+00:00,1.34,NaN
4,2018-11-01 03:00:00+00:00,1.34,NaN
...,...,...,...
45284,2023-12-31 19:00:00+00:00,10.30,NaN
45285,2023-12-31 20:00:00+00:00,10.30,NaN
45286,2023-12-31 21:00:00+00:00,10.30,NaN
45287,2023-12-31 22:00:00+00:00,10.30,NaN


## Run for all stations

In [25]:
for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DEG", add_missing=True)

    # check if gebiet is Maingebiet or Werragebiet
    if id in raw_meta_main["PegelID"].values:
        gebiet = "Maingebiet"
    elif id in raw_meta_werra["PegelID"].values:
        gebiet = "Werragebiet"
    elif id in raw_meta_werra_leine["PegelID"].values:
        gebiet = "Werra_Leinegebiet"
    elif id in raw_meta_saale["PegelID"].values:
        gebiet = "Saalegebiet"
    else:
        raise ValueError(f"Station {id} not found in either Maingebiet or Werragebiet metadata.")
    
    # parse data for the station
    data = parse_station_data(id, aggregate_hourly=True, gebiet=gebiet)

    # get the metadata of the station
    meta = raw_meta[raw_meta["PegelID"] == id]

    # get metadata of the station, if station is part of camelsp, use the camelsp metadata, otherwise use the metadata from raw_meta
    # parse the location
    if id in camelsp_meta_all["provider_id"].values:
        lon, lat = camelsp_meta_all[camelsp_meta_all["provider_id"] == id][["lon", "lat"]].values[0]
        easting, northing = camelsp_meta_all[camelsp_meta_all["provider_id"] == id][["x", "y"]].values[0]

        # other metadata fields
        gauge_name = camelsp_meta_all[camelsp_meta_all["provider_id"] == id]["gauge_name"].values[0]
        waterbody_name = camelsp_meta_all[camelsp_meta_all["provider_id"] == id]["waterbody_name"].values[0]
        gauge_elev_metadata = camelsp_meta_all[camelsp_meta_all["provider_id"] == id]["gauge_elevation"].values[0]
        area_metadata = camelsp_meta_all[camelsp_meta_all["provider_id"] == id]["area"].values[0]

        in_camelsp = True
    else:
        # location
        lon, lat = meta["lon"].values[0], meta["lat"].values[0]
        easting, northing = meta["easting"].values[0], meta["northing"].values[0]

        # other metadata fiels
        gauge_name = meta["Name"].values[0]
        waterbody_name = meta["Gewässer"].values[0]
        gauge_elev_metadata = meta["Pegelnull [DHHN2016]"].values[0]
        area_metadata = meta["AE [km²]"].values[0]

        in_camelsp = False
    
    # set metadata values from camelsp metadata
    station_meta = {
        "gauge_id": gauge_id,
        "provider_id": id,
        "gauge_name": gauge_name,
        "waterbody_name": waterbody_name,
        "federal_state": "Thüringen",
        "gauge_lon": lon,
        "gauge_lat": lat,
        "gauge_easting": easting,
        "gauge_northing": northing,
        "gauge_elev_metadata": gauge_elev_metadata,
        "area_metadata": area_metadata,
        "part_of_camelsp": in_camelsp,
    }
    
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(meta)
    

100%|██████████| 140/140 [16:13<00:00,  6.95s/it]
